# Chapter 8 explore: Fine-Tuning at Scale with Checkpointing and Experiment Tracking

Interactive companion to `code/chapter_08/finetune_at_scale.py`. Fine-tunes on Chapter 7's full training set with per-epoch checkpointing and CSV/JSONL metrics logging. The full run takes about 30 minutes on CPU -- this notebook uses a small subset so you can see the mechanism run quickly.

In [ ]:
import sys
sys.path.insert(0, "../code/chapter_01")
sys.path.insert(0, "../code/chapter_02")
sys.path.insert(0, "../code/chapter_03")
sys.path.insert(0, "../code/chapter_05")
sys.path.insert(0, "../code/chapter_06")
sys.path.insert(0, "../code/chapter_07")
sys.path.insert(0, "../code/chapter_08")

from load_local_model import MODEL_NAME, load_model_and_tokenizer
from build_training_examples import HELD_OUT_REPORT
from data_quality_gate import FULL_SET_DIR
from format_training_chunks import build_training_set_at_scale, build_timeline_examples_for_report
from first_lora_finetune import build_lora_model
from finetune_at_scale import MetricsLogger, new_run_dir, train_with_checkpoints, load_checkpoint

examples, skipped, artifacts = build_training_set_at_scale()
held_out_examples, _ = build_timeline_examples_for_report(FULL_SET_DIR / HELD_OUT_REPORT)
print(f"{len(examples)} training examples, {len(held_out_examples)} held-out examples")

model, tokenizer = load_model_and_tokenizer(MODEL_NAME)

Train on a small subset for 2 epochs, checkpointing and logging each one.

In [ ]:
run_dir = new_run_dir()
logger = MetricsLogger(run_dir)
print(f"Run directory: {run_dir}")

lora_model = build_lora_model(model)
train_with_checkpoints(lora_model, tokenizer, examples[:10], held_out_examples[:3], run_dir, logger, num_epochs=2)

Read the metrics back from disk -- this is exactly what a spreadsheet would show you.

In [ ]:
print((run_dir / "metrics.csv").read_text())

Simulate a crash and restart: reload the base model fresh, load only checkpoint_2 from disk, and confirm training resumes (loss keeps falling, not jumping back up).

In [ ]:
del model, lora_model
model, tokenizer = load_model_and_tokenizer(MODEL_NAME)
resumed_model = load_checkpoint(model, run_dir / "checkpoint_2")
train_with_checkpoints(resumed_model, tokenizer, examples[:10], held_out_examples[:3], run_dir, logger, num_epochs=1, start_epoch=2)